In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold

In [2]:
df = pd.read_parquet('result/train/02_train_승인매출정보_라벨인코딩.parquet')
df

,기준년월,ID,최종이용일자_기본,최종이용일자_신판,최종이용일자_CA,최종이용일자_카드론,최종이용일자_체크,최종이용일자_일시불,최종이용일자_할부,이용건수_신용_B0M,...,승인거절건수_한도초과_B0M,승인거절건수_BL_B0M,승인거절건수_입력오류_B0M,승인거절건수_기타_B0M,승인거절건수_R3M,승인거절건수_한도초과_R3M,승인거절건수_BL_R3M,승인거절건수_입력오류_R3M,승인거절건수_기타_R3M,이용금액대
0,201807,TRAIN_000000,20180719,20180713,20180719,10101,20180203,20180709,20180713,11,...,0,0,0,0,3,3,0,0,0,0
1,201807,TRAIN_000001,20180719,20180719,20170728,20170327,10101,20180719,20171231,13,...,0,0,0,0,3,3,0,0,0,2
2,201807,TRAIN_000002,20180706,20180706,20180706,20151119,20141230,20180706,20180627,12,...,0,0,0,0,0,0,0,0,0,0
3,201807,TRAIN_000003,20180721,20180715,20180721,10101,20141111,20180704,20180715,6,...,0,0,0,0,3,3,0,0,0,0
4,201807,TRAIN_000004,20180124,20180124,10101,10101,20180512,20180124,10101,-2,...,0,0,0,0,0,0,0,0,0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2399995,201812,TRAIN_399995,20181220,20181220,10101,10101,20181212,20181220,20160501,2,...,0,0,0,0,0,0,0,0,0,5
2399996,201812,TRAIN_399996,20181202,20181202,10101,20170112,10101,20181202,20180112,10,...,0,0,0,0,0,0,0,0,0,0
2399997,201812,TRAIN_399997,20181230,20181230,10101,10101,20131124,20181230,20180919,10,...,0,0,0,0,0,0,0,0,0,1
2399998,201812,TRAIN_399998,20161224,20161224,10101,10101,10101,20161224,20150122,-2,...,0,0,0,0,0,0,0,0,0,5


In [3]:
# 상수값 컬럼 제거
def remove_constant_columns(df):
    return df.loc[:, df.nunique() > 1]

# 결측치 비율 기준 컬럼 제거
def remove_high_na_columns(df, threshold=0.2):
    null_ratio = df.isnull().mean()
    to_drop = null_ratio[null_ratio > threshold].index
    print(f"⤷ 결측치 {int(threshold * 100)}% 초과 컬럼 수: {len(to_drop)}")
    return df.drop(columns=to_drop)

# 낮은 분산 컬럼 제거
def remove_low_variance_columns(df, threshold=0.01):
    selector = VarianceThreshold(threshold=threshold)
    reduced = selector.fit_transform(df)
    return df.loc[:, selector.get_support()]

# 상관관계 높은 컬럼 제거
def remove_highly_correlated(df, threshold=0.6):
    corr_matrix = df.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
    print(f"⤷ 상관관계 {threshold} 초과 컬럼 수: {len(to_drop)}")
    return df.drop(columns=to_drop), to_drop

# 전체 전처리 실행 함수
def preprocess_by_correlation_only(df_numeric, na_thresh=0.2, var_thresh=0.01, corr_thresh=0.6):
    print(" 초기 수치형 컬럼 수:", df_numeric.shape[1])

    df1 = remove_constant_columns(df_numeric)
    print(" 상수값 컬럼 제거 후:", df1.shape[1])

    df2 = remove_high_na_columns(df1, threshold=na_thresh)
    print(" 결측치 컬럼 제거 후:", df2.shape[1])

    df3 = remove_low_variance_columns(df2, threshold=var_thresh)
    print(" 낮은 분산 컬럼 제거 후:", df3.shape[1])

    df_final, dropped_corr_cols = remove_highly_correlated(df3, threshold=corr_thresh)
    print(" 상관관계 중복 제거 후:", df_final.shape[1])

    return df_final, dropped_corr_cols

In [5]:
# exclude 목록 정의
exclude_cols = ['ID']

# ID 컬럼 따로 저장
df_id = df[exclude_cols] 

# 수치형 컬럼만 추출 (ID 제외된 상태로 전처리용 데이터 준비)
df_numeric = df.select_dtypes(include=[np.number])

# 전처리 실행
df_cleaned, dropped_corr_cols = preprocess_by_correlation_only(df_numeric)

# 전처리 완료된 결과에 ID 다시 붙이기
df_cleaned = pd.concat([df_id, df_cleaned], axis=1)

# 결과 확인
print("최종 남은 컬럼 수:", df_cleaned.shape[1])
print("상관관계로 제거된 컬럼 수:", len(dropped_corr_cols))
print("제거된 컬럼 목록:", dropped_corr_cols[:10])

 초기 수치형 컬럼 수: 405
 상수값 컬럼 제거 후: 374
⤷ 결측치 20% 초과 컬럼 수: 2
 결측치 컬럼 제거 후: 372
 낮은 분산 컬럼 제거 후: 341
⤷ 상관관계 0.6 초과 컬럼 수: 270
 상관관계 중복 제거 후: 71
최종 남은 컬럼 수: 72
상관관계로 제거된 컬럼 수: 270
제거된 컬럼 목록: ['최종이용일자_신판', '최종이용일자_일시불', '이용건수_신판_B0M', '이용건수_일시불_B0M', '이용건수_할부_무이자_B0M', '이용금액_일시불_B0M', '이용금액_할부_B0M', '이용금액_할부_유이자_B0M', '이용금액_할부_무이자_B0M', '이용금액_CA_B0M']


In [6]:
df_cleaned

,ID,기준년월,최종이용일자_기본,최종이용일자_CA,최종이용일자_카드론,최종이용일자_체크,최종이용일자_할부,이용건수_신용_B0M,이용건수_할부_B0M,이용건수_할부_유이자_B0M,...,이용개월수_C페이_R6M,이용개월수_D페이_R6M,이용금액_D페이_B0M,이용개월수_선결제_R6M,이용횟수_연체_R6M,가맹점매출금액_B1M,건수_할부전환_R6M,승인거절건수_R3M,승인거절건수_BL_R3M,승인거절건수_기타_R3M
0,TRAIN_000000,201807,20180719,20180719,10101,20180203,20180713,11,1,0,...,0,0,0,0,1,0,0,3,0,0
1,TRAIN_000001,201807,20180719,20170728,20170327,10101,20171231,13,0,0,...,0,0,0,0,0,0,0,3,0,0
2,TRAIN_000002,201807,20180706,20180706,20151119,20141230,20180627,12,0,0,...,0,0,0,0,0,6190,0,0,0,0
3,TRAIN_000003,201807,20180721,20180721,10101,20141111,20180715,6,1,1,...,0,0,0,0,0,0,0,3,0,0
4,TRAIN_000004,201807,20180124,10101,10101,20180512,10101,-2,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2399995,TRAIN_399995,201812,20181220,10101,10101,20181212,20160501,2,0,0,...,0,0,0,0,0,0,0,0,0,0
2399996,TRAIN_399996,201812,20181202,10101,20170112,10101,20180112,10,0,0,...,0,0,0,0,0,0,0,0,0,0
2399997,TRAIN_399997,201812,20181230,10101,10101,20131124,20180919,10,0,0,...,0,0,0,0,0,0,0,0,0,0
2399998,TRAIN_399998,201812,20161224,10101,10101,10101,20150122,-2,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
df_cleaned.to_parquet("result/train/03_train_승인매출정보_전처리.parquet", index=False)